## baseline:

In [7]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.experiment import run_experiment
from src.model import CNNBaseline
from src.train import evaluate, fit, make_scheduler, train_one_epoch
from src.utils import get_device, set_seed


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEED = 42
set_seed(SEED)
device = get_device()

IMG_SIZE = 128
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('../data/processed/train', transform=base_transform)
val_ds = datasets.ImageFolder('../data/processed/val', transform=base_transform)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
device= get_device()
x , y = next(iter(train_loader))
print(x.shape , y.shape)
model= CNNBaseline().to(device)
print(model(x.to(device)).shape)
print(sum(p.numel() for p in model.parameters()))

In [ ]:
x, y = next(iter(train_loader))
x, y = x.to(device), y.to(device)
m = CNNBaseline().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
for step in range(100):
    opt.zero_grad()
    loss = crit(m(x), y)
    loss.backward()
    opt.step()
    if step % 20 == 0:
        print(step, loss.item())

## training loop :

In [ ]:
EPOCHS = 20
model = CNNBaseline().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=1e-3)
meta = {
    'architecture': 'CNNBaseline',
    'class_to_idx': train_ds.class_to_idx ,
    'IMG_SIZE' : IMG_SIZE,
    'seed': SEED,
    'config': { 'epochs': EPOCHS , 'lr': 1e-3 , 'batch_size': 32 , 'optimizer':'Adam'}}
history = fit(model , train_loader , val_loader , optimizer , criterion , device , EPOCHS , '../checkpoints/baseline.pt' , '../logs/baseline.csv' , meta = meta)

In [8]:
run_experiment('baseline_v2', device, paths, epochs=20)

{'epoch': 1, 'train_loss': 1.935445547033302, 'val_loss': 1.8202717695957662, 'val_macro_f1': 0.21016141186526996, 'lr': 0.001}
{'epoch': 2, 'train_loss': 1.766879270103572, 'val_loss': 1.7503060243372095, 'val_macro_f1': 0.2416910812724779, 'lr': 0.001}
{'epoch': 3, 'train_loss': 1.6726113377131253, 'val_loss': 1.670717086352355, 'val_macro_f1': 0.326422405914253, 'lr': 0.001}
{'epoch': 4, 'train_loss': 1.5752227990990069, 'val_loss': 1.5219523531888957, 'val_macro_f1': 0.3782759149296867, 'lr': 0.001}
{'epoch': 5, 'train_loss': 1.4940714282774938, 'val_loss': 1.4753119257888614, 'val_macro_f1': 0.3914153885903112, 'lr': 0.001}
{'epoch': 6, 'train_loss': 1.4366343121616145, 'val_loss': 1.4281534100255222, 'val_macro_f1': 0.43272829659970763, 'lr': 0.001}
{'epoch': 7, 'train_loss': 1.4177312850952148, 'val_loss': 1.4041599604536736, 'val_macro_f1': 0.41926216952531625, 'lr': 0.001}
{'epoch': 8, 'train_loss': 1.3388994824187581, 'val_loss': 1.3158772997822321, 'val_macro_f1': 0.45872479

{'macro_precision': 0.6124559077408441,
 'macro_recall': 0.5996137530749213,
 'macro_f1': 0.5881403971198502,
 'weighted_precision': 0.644597745282378,
 'weighted_recall': 0.6548463356973995,
 'weighted_f1': 0.6359526179525917,
 'accuracy': 0.6548463356973995,
 'lowest_recall_class': 'vanet',
 'lowest_precision_class': 'vanet',
 'lowest_f1_class': 'vanet',
 'highest_f1_class': 'taxi',
 'recall_ambulance': np.float64(0.6296296296296297),
 'precision_ambulance': np.float64(0.9444444444444444),
 'f1_ambulance': np.float64(0.7555555555555555),
 'support_ambulance': 54,
 'recall_autobus': np.float64(0.6610169491525424),
 'precision_autobus': np.float64(0.65),
 'f1_autobus': np.float64(0.6554621848739496),
 'support_autobus': 59,
 'recall_kamyun': np.float64(0.7166666666666667),
 'precision_kamyun': np.float64(0.524390243902439),
 'f1_kamyun': np.float64(0.6056338028169014),
 'support_kamyun': 60,
 'recall_kamyunet': np.float64(0.2413793103448276),
 'precision_kamyunet': np.float64(0.3783783